# Task 3 ensemble

Rating prediction with timestamps. The Task 1 models (timestamps ignored) plus four temporal models:

- **time-aware bias model**: time-bin, user-drift and user-period biases in closed form (PML Sec. 7.2.2, Koren 2009)
- **timeSVD++**: SVD++ on top of the time-aware bias model
- **time-decayed item kNN**: neighbours rated close to the asked day count more
- **temporal zero model**: P(0) also from the zeros the user gave around that day

Every sub-model cell trains one model and stores it in `trained_models/task3/<model>.pt`.
The blend cells only need the stored files, so after a kernel restart nothing has to be retrained.
All hyperparameters: `submission/lib/config.py`, Task 3 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.config import TASK3_LIGHTGBM_BLENDER, TASK3_RIDGE_BLENDER, TASK3_SUBMITTED_BLENDER
from lib.models.bias_model import load_bias_model, store_bias_model
from lib.models.blender import load_blender, store_blender
from lib.models.item_knn import load_item_knn, store_item_knn
from lib.models.SVDpp import load_svd, load_svdpp, store_svd, store_svdpp
from lib.models.time_bias_model import load_time_bias_model, store_time_bias_model
from lib.models.time_item_knn import load_time_item_knn, store_time_item_knn
from lib.models.time_svdpp import load_time_svdpp, store_time_svdpp
from lib.models.time_zero_model import load_time_zero_model, store_time_zero_model
from lib.models.zero_model import load_zero_model, store_zero_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


In [2]:
# training file + test file with timestamps -> task3.test_days is set
task3 = assignment1.load_task3()

## Sub-models without time

In [3]:
# bias model: mu + b_u + b_i in closed form (seconds)
bias_model_result = assignment1.train_task3_bias_model(task3)
store_bias_model(bias_model_result, "task3")

[23:41:34] bias model | valid MSE 0.9138
[23:41:34] stored bias_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\bias_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/bias_model.pt')

In [4]:
# SVD (a few minutes on the RTX 5080)
svd_result = assignment1.train_task3_svd(task3)
store_svd(svd_result, "task3")

[23:41:36] epoch   0 | train MSE 0.8484 | valid MSE 0.9138  (starting point)
[23:41:47] epoch   1 | train MSE 0.8398 | valid MSE 0.9128
[23:41:58] epoch   2 | train MSE 0.7866 | valid MSE 0.9031
[23:42:09] epoch   3 | train MSE 0.6868 | valid MSE 0.8933
[23:42:20] epoch   4 | train MSE 0.5856 | valid MSE 0.8900
[23:42:30] epoch   5 | train MSE 0.5016 | valid MSE 0.8908
[23:42:41] epoch   6 | train MSE 0.4349 | valid MSE 0.8938
[23:42:52] epoch   7 | train MSE 0.3827 | valid MSE 0.8972
[23:43:03] epoch   8 | train MSE 0.3421 | valid MSE 0.9009
[23:43:13] epoch   9 | train MSE 0.3109 | valid MSE 0.9042
[23:43:24] epoch  10 | train MSE 0.2870 | valid MSE 0.9070
[23:43:24] best valid MSE 0.8900 at epoch 4
[23:43:36] refit epoch   1 | train MSE 0.8542
[23:43:48] refit epoch   2 | train MSE 0.8357
[23:44:00] refit epoch   3 | train MSE 0.7671
[23:44:11] refit epoch   4 | train MSE 0.6704
[23:44:11] stored svd -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_20

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/svd.pt')

In [5]:
# SVD++ (a few minutes on the RTX 5080)
svdpp_result = assignment1.train_task3_svdpp(task3)
store_svdpp(svdpp_result, "task3")

[23:44:13] epoch   0 | train MSE 0.8484 | valid MSE 0.9138  (starting point)
[23:44:15] epoch   1 | train MSE 0.7833 | valid MSE 0.8986
[23:44:18] epoch   2 | train MSE 0.6127 | valid MSE 0.8861
[23:44:21] epoch   3 | train MSE 0.4054 | valid MSE 0.8931
[23:44:24] epoch   4 | train MSE 0.2620 | valid MSE 0.9122
[23:44:27] epoch   5 | train MSE 0.1844 | valid MSE 0.9280
[23:44:30] epoch   6 | train MSE 0.1439 | valid MSE 0.9416
[23:44:32] epoch   7 | train MSE 0.1222 | valid MSE 0.9514
[23:44:36] epoch   8 | train MSE 0.1098 | valid MSE 0.9602
[23:44:36] best valid MSE 0.8861 at epoch 2
[23:44:40] refit epoch   1 | train MSE 0.8495
[23:44:43] refit epoch   2 | train MSE 0.7685
[23:44:44] stored svdpp -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\svdpp.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/svdpp.pt')

In [6]:
# item-item kNN on the bias model's residuals (CPU, about a minute)
item_knn_result = assignment1.train_task3_item_knn(task3)
store_item_knn(item_knn_result, "task3")

[23:44:50] item kNN | 858,888 neighbour links | valid MSE 0.8997
[23:44:56] stored item_knn -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\item_knn.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/item_knn.pt')

In [7]:
# zero model: P(rating = 0) (CPU, seconds)
zero_model_result = assignment1.train_task3_zero_model(task3)
store_zero_model(zero_model_result, "task3")

[23:44:57] zero model | valid AUC 0.8661 | log loss 0.0564 | mean P(0) 0.0169 vs share 0.0164
[23:44:57] stored zero_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\zero_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/zero_model.pt')

## Temporal sub-models

In [8]:
# time-aware bias model: mu + b_u + b_i + b_bin(t) + b_{i,bin(t)} + alpha_u dev_u(t) + b_{u,period(t)} in closed form (CPU, seconds)
time_bias_model_result = assignment1.train_task3_time_bias_model(task3)
store_time_bias_model(time_bias_model_result, "task3")

[23:45:00] time bias model | valid MSE 0.8829
[23:45:02] stored time_bias_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\time_bias_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/time_bias_model.pt')

In [9]:
# timeSVD++: SVD++ on top of the time-aware bias model (the slowest sub-model, GPU)
time_svdpp_result = assignment1.train_task3_time_svdpp(task3)
store_time_svdpp(time_svdpp_result, "task3")

[23:45:06] epoch   0 | train MSE 0.6747 | valid MSE 0.8829  (starting point)
[23:45:09] epoch   1 | train MSE 0.6259 | valid MSE 0.8708
[23:45:13] epoch   2 | train MSE 0.4956 | valid MSE 0.8605
[23:45:16] epoch   3 | train MSE 0.3337 | valid MSE 0.8661
[23:45:19] epoch   4 | train MSE 0.2219 | valid MSE 0.8818
[23:45:21] epoch   5 | train MSE 0.1616 | valid MSE 0.8947
[23:45:25] epoch   6 | train MSE 0.1298 | valid MSE 0.9059
[23:45:29] epoch   7 | train MSE 0.1126 | valid MSE 0.9135
[23:45:32] epoch   8 | train MSE 0.1023 | valid MSE 0.9205
[23:45:32] best valid MSE 0.8605 at epoch 2
[23:45:38] refit epoch   1 | train MSE 0.6775
[23:45:41] refit epoch   2 | train MSE 0.6184
[23:45:42] stored time_svdpp -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\time_svdpp.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/time_svdpp.pt')

In [10]:
# time-decayed item kNN: neighbour weight * (floor + (1 - floor) exp(-|t - t_uj| / time_decay_days)) (CPU, about a minute)
time_item_knn_result = assignment1.train_task3_time_item_knn(task3)
store_time_item_knn(time_item_knn_result, "task3")

[23:45:48] time item kNN | decay 365 days, floor 0.5 | valid MSE 0.9006
[23:45:55] stored time_item_knn -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\time_item_knn.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/time_item_knn.pt')

In [11]:
# temporal zero model: P(rating = 0) + the user's zeros on the same day / within a month (CPU, seconds)
time_zero_model_result = assignment1.train_task3_time_zero_model(task3)
store_time_zero_model(time_zero_model_result, "task3")

[23:45:56] time zero model | valid AUC 0.8725 | log loss 0.0538 | mean P(0) 0.0168 vs share 0.0164
[23:45:57] stored time_zero_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\time_zero_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/time_zero_model.pt')

## Blend

In [12]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "bias_model": load_bias_model,
    "svd": load_svd,
    "svdpp": load_svdpp,
    "item_knn": load_item_knn,
    "zero_model": load_zero_model,
    "time_bias_model": load_time_bias_model,
    "time_svdpp": load_time_svdpp,
    "time_item_knn": load_time_item_knn,
    "time_zero_model": load_time_zero_model,
}
task3_model_results = {model_name: sub_model_loaders[model_name]("task3") for model_name in TASK3_SUBMITTED_BLENDER.blended_models}

In [13]:
# ridge regression: learned weighted average of the sub-model predictions
ridge_blend_result = assignment1.train_task3_blender(task3, task3_model_results, TASK3_RIDGE_BLENDER)
store_blender(ridge_blend_result, "task3")

[23:45:59] bias_model             valid MSE 0.9138
[23:45:59] svd                    valid MSE 0.8900
[23:45:59] svdpp                  valid MSE 0.8861
[23:45:59] item_knn               valid MSE 0.8997
[23:45:59] time_bias_model        valid MSE 0.8829
[23:45:59] time_svdpp             valid MSE 0.8605
[23:45:59] time_item_knn          valid MSE 0.9006
[23:45:59] ridge blend                valid MSE 0.8467  (out-of-fold, 5 folds)
[23:45:59] ridge blend inputs: bias_model -2.31, svd 0.0739, svdpp 0.151, item_knn 0.718, zero_model 1.81, time_bias_model 0.164, time_svdpp 0.624, time_item_knn 1.55, time_zero_model -1.8
[23:45:59] stored ridge_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\ridge_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/ridge_blender.pt')

In [14]:
# LightGBM on the sub-model predictions + #ratings of user and item + when the rating happened (time features)
lightgbm_blend_result = assignment1.train_task3_blender(task3, task3_model_results, TASK3_LIGHTGBM_BLENDER)
store_blender(lightgbm_blend_result, "task3")

[23:46:03] bias_model             valid MSE 0.9138
[23:46:03] svd                    valid MSE 0.8900
[23:46:03] svdpp                  valid MSE 0.8861
[23:46:03] item_knn               valid MSE 0.8997
[23:46:03] time_bias_model        valid MSE 0.8829
[23:46:03] time_svdpp             valid MSE 0.8605
[23:46:03] time_item_knn          valid MSE 0.9006
[23:46:03] lightgbm blend                valid MSE 0.8498  (out-of-fold, 5 folds)
[23:46:03] lightgbm blend inputs: bias_model 997, svd 829, svdpp 735, item_knn 548, zero_model 876, time_bias_model 603, time_svdpp 1.02e+03, time_item_knn 341, time_zero_model 984, log_user_ratings 734, log_item_ratings 1.04e+03, log_same_day_ratings 55, log_window_ratings 417, log_earlier_ratings 749, item_age_years 1.06e+03, years_since_2000 1.01e+03
[23:46:03] stored lightgbm_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task3\lightgbm_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task3/lightgbm_blender.pt')

In [15]:
# the submitted blend (TASK3_SUBMITTED_BLENDER in lib/config.py) writes submission/output/task_3_predictions.csv
# -> set it to whichever blend printed the lower out-of-fold MSE above
submitted_blend_result = load_blender("task3", TASK3_SUBMITTED_BLENDER.method)
assignment1.write_task3_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK3_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_3_predictions.csv
